In [1]:
R.version.string
library(limma)
cat("limma OK:", as.character(packageVersion("limma")), "\n")

[1] "R version 4.5.3 (2026-03-11)"

limma OK: 3.66.0 


In [2]:
base <- "~/python_practice/luad_project/results"

# đọc ma trận: cột 1 (probe) làm tên hàng
expr <- read.csv(file.path(base, "expression_matrix_log2.csv"), row.names = 1)
expr <- as.matrix(expr)              # limma cần MA TRẬN, không phải data.frame

dim(expr)                            # kỳ vọng: 22283   107
expr[1:3, 1:3]                       # xem góc trên-trái

[1] 22283   107

,GSM254625,GSM254626,GSM254627
1007_s_at,10.927084,10.416978,10.628538
1053_at,6.895217,6.924856,7.550245
117_at,8.110190,7.760228,7.974676


In [3]:
info <- read.csv(file.path(base, "sample_info.csv"), row.names = 1)

dim(info)                    # kỳ vọng: 107   2
head(info, 3)                # xem cột patient, tissue

# ── KIỂM SỐNG CÒN: thứ tự mẫu 2 file có khớp không? ──
all(rownames(info) == colnames(expr))    # phải ra TRUE

[1] 107   2

,patient,tissue
,<chr>,<chr>
GSM254625,GT00006,Lung Tumor
GSM254626,GT00006,Normal Lung
GSM254627,GT00007,Lung Tumor


[1] TRUE

In [4]:
# gom tên loại mô về gọn: "Lung Tumor" -> Tumor, "Normal Lung" -> Normal
tissue <- ifelse(info$tissue == "Lung Tumor", "Tumor", "Normal")

# biến thành factor, ÉP Normal làm mốc (baseline) đứng trước
tissue <- factor(tissue, levels = c("Normal", "Tumor"))

table(tissue)          # đếm mỗi nhóm — kỳ vọng Normal 49, Tumor 58
levels(tissue)         # kỳ vọng: "Normal" "Tumor"  (Normal TRƯỚC = baseline)

# dựng design matrix
design <- model.matrix(~ tissue)
head(design, 3)        # xem 3 hàng đầu
colnames(design)       # kỳ vọng: "(Intercept)"  "tissueTumor"

tissue
Normal  Tumor 
    49     58 

[1] "Normal" "Tumor"

,(Intercept),tissueTumor
1,1,1
2,1,0
3,1,1


[1] "(Intercept)" "tissueTumor"

In [5]:
patient <- factor(info$patient)      # mã bệnh nhân làm nhân tố nhóm cặp
length(levels(patient))              # số bệnh nhân — kỳ vọng 74

[1] 74

In [6]:
corfit <- duplicateCorrelation(expr, design, block = patient)
corfit$consensus.correlation        # một con số — mức tương quan trung bình

[1] 0.2214803

In [7]:
# 1) khớp mô hình tuyến tính, CÓ báo cho limma biết cấu trúc cặp
fit <- lmFit(expr, design, block = patient, correlation = corfit$consensus.correlation)

# 2) empirical Bayes: làm mượt phương sai xuyên gene (sức mạnh của limma)
fit <- eBayes(fit)

# 3) xem kết quả sơ bộ cho hệ số "tissueTumor"
head(topTable(fit, coef = "tissueTumor", n = 5))

,logFC,AveExpr,t,P.Value,adj.P.Val,B
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
209074_s_at,-3.370091,8.995179,-26.33333,6.307042e-49,1.405398e-44,100.93712
204396_s_at,-2.388091,8.677189,-23.93815,4.337413e-45,4.832528e-41,92.21556
209555_s_at,-2.410301,8.261099,-23.79346,7.542438e-45,5.346285e-41,91.66860
204677_at,-2.607152,8.004204,-23.73065,9.597066e-45,5.346285e-41,91.43041
206209_s_at,-2.545589,8.082080,-23.61408,1.502390e-44,6.695553e-41,90.98724


In [1]:
library(limma)
base <- "~/python_practice/luad_project/results"

# đọc lại dữ liệu
expr <- as.matrix(read.csv(file.path(base, "expression_matrix_log2.csv"), row.names = 1))
info <- read.csv(file.path(base, "sample_info.csv"), row.names = 1)
stopifnot(all(rownames(info) == colnames(expr)))   # cổng sống còn — phải khớp

# dựng lại design + block
tissue  <- factor(ifelse(info$tissue == "Lung Tumor", "Tumor", "Normal"),
                  levels = c("Normal", "Tumor"))
patient <- factor(info$patient)
design  <- model.matrix(~ tissue)

# mô hình ghép cặp (bước chậm)
corfit <- duplicateCorrelation(expr, design, block = patient)
fit <- lmFit(expr, design, block = patient, correlation = corfit$consensus.correlation)
fit <- eBayes(fit)
res_limma <- topTable(fit, coef = "tissueTumor", number = Inf, sort.by = "none")

# kiểm nhanh mọi thứ nạp đúng
cat("expr:", dim(expr), "| corr:", round(corfit$consensus.correlation, 3),
    "| res_limma:", dim(res_limma), "\n")

expr: 22283 107 | corr: 0.221 | res_limma: 22283 6 


In [2]:
py <- read.csv(file.path(base, "all_genes_results.csv"))
colnames(py)                                   # có 'probe','gene','mean_expr' không?
nrow(py)                                        # 12548 (mức gene) hay 22283 (mức probe)?
length(unique(py$gene))                         # số gene khác nhau
length(unique(py$probe))                        # số probe khác nhau

[1] "probe"        "log2FC"       "p_value"      "p_adj"        "neg_log_padj"
[6] "status"       "gene"         "mean_expr"    "nef_log_padj"

[1] 12548

[1] 12548

[1] 12548

In [3]:
res_limma$probe <- rownames(res_limma)    # tạo cột probe từ tên hàng
head(res_limma, 2)                         # kiểm: giờ có cột 'probe'

,logFC,AveExpr,t,P.Value,adj.P.Val,B,probe
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1007_s_at,0.4596665,10.560596,8.280418,3.509841e-13,4.198056e-12,19.215202,1007_s_at
1053_at,0.2534388,6.872995,6.370482,4.676956e-09,2.951476e-08,9.841865,1053_at


In [4]:
# chỉ lấy 2 cột cần từ py: probe + gene (để gắn tên gene)
map <- py[, c("probe", "gene")]

# ghép: giữ chỉ probe có mặt ở CẢ hai (tức 12548 probe đại diện)
merged <- merge(res_limma, map, by = "probe")

dim(merged)                 # kỳ vọng 12548   8  (6 cột limma + probe + gene)
head(merged, 3)

[1] 12548     8

,probe,logFC,AveExpr,t,P.Value,adj.P.Val,B,gene
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,1053_at,0.25343880,6.872995,6.3704818,4.676956e-09,2.951476e-08,9.841865,RFC2
2,121_at,0.01489150,9.871100,0.4789941,6.329072e-01,7.224194e-01,-7.376258,PAX8
3,1487_at,0.09505214,8.463736,3.0413605,2.953576e-03,7.097437e-03,-3.026624,ESRRA


In [5]:
# ghép theo gene; đánh dấu cột nào của limma, cột nào của python
cmp <- merge(merged, py, by = "gene", suffixes = c("_limma", "_py"))

dim(cmp)                    # kỳ vọng 12548  (nhiều cột)
# lấy ra đúng các cột cần để nhìn cho gọn
cmp_slim <- cmp[, c("gene", "logFC", "adj.P.Val", "log2FC", "p_adj")]
colnames(cmp_slim) <- c("gene", "lfc_limma", "padj_limma", "lfc_py", "padj_py")
head(cmp_slim, 3)

[1] 12548    16

,gene,lfc_limma,padj_limma,lfc_py,padj_py
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,A1CF,-0.004532477,9.587554e-01,-0.005406154,9.569482e-01
2,A2M,-1.347904605,2.136350e-23,-1.358073228,7.961513e-21
3,A4GALT,0.008012204,9.083179e-01,0.002289220,9.780943e-01


In [6]:
controls <- c("NEK2", "TTK", "PRC1", "AGER")
cmp_slim[cmp_slim$gene %in% controls, ]

,gene,lfc_limma,padj_limma,lfc_py,padj_py
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
296,AGER,-4.406440,3.372479e-40,-4.417470,2.847356e-36
7144,NEK2,1.487006,2.398236e-17,1.515365,2.083891e-15
8476,PRC1,1.383955,1.779101e-14,1.410437,8.240263e-13
11619,TTK,1.128533,4.151910e-13,1.151624,1.061006e-11


In [7]:
cor(cmp_slim$lfc_limma, cmp_slim$lfc_py)      # Pearson, trên cả 12548 gene

[1] 0.9997822

In [8]:
# gắn nhãn DEG cho từng phương pháp, cùng ngưỡng
deg_limma <- cmp_slim$padj_limma < 0.001 & abs(cmp_slim$lfc_limma) > 0.585
deg_py    <- cmp_slim$padj_py    < 0.001 & abs(cmp_slim$lfc_py)    > 0.585

cat("DEG limma:", sum(deg_limma), "\n")
cat("DEG python:", sum(deg_py), "\n")
cat("Trùng cả hai:", sum(deg_limma & deg_py), "\n")
cat("Chỉ limma:", sum(deg_limma & !deg_py), "\n")
cat("Chỉ python:", sum(!deg_limma & deg_py), "\n")

DEG limma: 1461 
DEG python: 1464 
Trùng cả hai: 1438 
Chỉ limma: 23 
Chỉ python: 26 


In [9]:
panel <- c("HLA-A","HLA-B","HLA-C","B2M","TAP1","TAP2",
           "TAPBP","PSMB8","PSMB9","NLRC5","IRF1","STAT1")

# lọc bảng limma xuống panel, lấy cột cần
panel_limma <- merged[merged$gene %in% panel,
                      c("gene", "logFC", "adj.P.Val")]
panel_limma                      # xem toàn bộ (ít dòng)

,gene,logFC,adj.P.Val
,<chr>,<dbl>,<dbl>
229,STAT1,0.50493435,1.011334e-05
1049,TAP1,0.17679865,1.627898e-01
1189,IRF1,-0.54548380,1.473459e-06
2316,PSMB9,0.03500510,8.288508e-01
2636,TAP2,0.17079135,1.889920e-02
5264,HLA-C,-0.16797355,8.794322e-04
5275,TAPBP,0.09210512,2.751179e-01
5376,PSMB8,0.24853082,1.434537e-02
5429,HLA-B,-0.15092606,3.518334e-02


In [10]:
panel_limma$pass <- panel_limma$adj.P.Val < 0.001 & abs(panel_limma$logFC) > 0.585
cat("Panel gene đạt ngưỡng (limma):", sum(panel_limma$pass), "/", nrow(panel_limma), "\n")

Panel gene đạt ngưỡng (limma): 0 / 11 


In [11]:
# đếm mỗi bệnh nhân xuất hiện mấy lần trong 107 mẫu
tab <- table(info$patient)

# bệnh nhân CÓ CẶP = xuất hiện đúng 2 lần (1 u + 1 lành)
paired_ids <- names(tab)[tab == 2]
length(paired_ids)                         # kỳ vọng 33

# mask chọn mẫu thuộc nhóm có cặp
keep <- info$patient %in% paired_ids
sum(keep)                                   # kỳ vọng 66 (33 cặp × 2)

[1] 33

[1] 66

In [12]:
expr_p    <- expr[, keep]                        # chỉ 66 cột (mẫu có cặp)
tissue_p  <- droplevels(tissue[keep])            # loại/giữ mức, bỏ mức thừa
patient_p <- droplevels(patient[keep])           # còn đúng 33 mức bệnh nhân

dim(expr_p)                                       # kỳ vọng 22283  66
length(levels(patient_p))                         # kỳ vọng 33
table(tissue_p)                                   # kỳ vọng Normal 33, Tumor 33

[1] 22283    66

[1] 33

tissue_p
Normal  Tumor 
    33     33 

In [13]:
design_p <- model.matrix(~ patient_p + tissue_p)   # patient là fixed effect
dim(design_p)                                        # kỳ vọng 66  34  (1 intercept +32 patient +1 tissue)

fit_p <- lmFit(expr_p, design_p)                     # KHÔNG cần block/correlation nữa
fit_p <- eBayes(fit_p)
res_p <- topTable(fit_p, coef = "tissue_pTumor", number = Inf, sort.by = "none")
res_p$probe <- rownames(res_p)
dim(res_p)                                            # kỳ vọng 22283  7

[1] 66 34

[1] 22283     7

In [14]:
# gắn gene vào res_p (dùng map probe->gene đã có từ py)
res_p_gene <- merge(res_p, py[, c("probe","gene")], by = "probe")

controls <- c("NEK2","TTK","PRC1","AGER")
panel    <- c("HLA-A","HLA-B","HLA-C","B2M","TAP1","TAP2",
              "TAPBP","PSMB8","PSMB9","NLRC5","IRF1","STAT1")

# đối chứng: xem dấu còn đúng không
res_p_gene[res_p_gene$gene %in% controls, c("gene","logFC","adj.P.Val")]

,gene,logFC,adj.P.Val
,<chr>,<dbl>,<dbl>
2553,NEK2,1.335474,1.126422e-09
2671,TTK,1.003713,5.908706e-07
5943,AGER,-4.350034,9.685472e-18
9073,PRC1,1.241462,2.638496e-08


In [15]:
panel_p <- res_p_gene[res_p_gene$gene %in% panel, c("gene","logFC","adj.P.Val")]
panel_p$pass <- panel_p$adj.P.Val < 0.001 & abs(panel_p$logFC) > 0.585
cat("Panel đạt ngưỡng (sensitivity):", sum(panel_p$pass), "/", nrow(panel_p), "\n")
panel_p

Panel đạt ngưỡng (sensitivity): 1 / 11 


,gene,logFC,adj.P.Val,pass
,<chr>,<dbl>,<dbl>,<lgl>
229,STAT1,0.483574751,0.0027661038,FALSE
1049,TAP1,0.176532337,0.2827821399,FALSE
1189,IRF1,-0.598892295,0.0001050108,TRUE
2316,PSMB9,-0.003847585,0.9843766602,FALSE
2636,TAP2,0.175875876,0.0452688202,FALSE
5264,HLA-C,-0.164475079,0.0089344558,FALSE
5275,TAPBP,0.065416817,0.5230223168,FALSE
5376,PSMB8,0.231748788,0.0394521406,FALSE
5429,HLA-B,-0.128017158,0.1421863072,FALSE


In [16]:
# 1) bảng đầy đủ: gene + thống kê limma, sắp theo p
out_full <- merged[, c("gene","probe","logFC","AveExpr","t","P.Value","adj.P.Val","B")]
out_full <- out_full[order(out_full$adj.P.Val), ]        # sắp tăng dần theo FDR
write.csv(out_full, file.path(base, "limma_paired_gene_level.csv"), row.names = FALSE)

# 2) bảng panel (dùng panel_limma đã có từ 2.5, + cột pass)
write.csv(panel_limma, file.path(base, "limma_antigen_panel.csv"), row.names = FALSE)

# kiểm đã ghi
list.files(base, pattern = "limma")

[1] "limma_antigen_panel.csv"     "limma_paired_gene_level.csv"

In [19]:
panel_limma <- merged[merged$gene %in% c("HLA-A","HLA-B","HLA-C","B2M","TAP1","TAP2","TAPBP","PSMB8","PSMB9","NLRC5","IRF1","STAT1"), c("gene","logFC","adj.P.Val")]
panel_limma$pass <- panel_limma$adj.P.Val < 0.001 & abs(panel_limma$logFC) > 0.585

In [22]:
repo <- "~/python_practice/luad_project"           # GỐC repo, không phải results/
file.exists(file.path(repo, "README.md"))           # kỳ vọng TRUE

[1] TRUE

In [23]:
repo <- "~/python_practice/luad_project"
readme <- readLines(file.path(repo, "README.md"))
any(grepl("Paired re-analysis", readme))     # ← mình cần con số này
grep("___", readme, value = TRUE)             # ← và cái này: còn sót ___ không

Warning message in readLines(file.path(repo, "README.md")):
“incomplete final line found on '~/python_practice/luad_project/README.md'”


[1] TRUE

character(0)

In [24]:
grep("___", readme, value = TRUE)

character(0)

In [25]:
# in từ dòng chứa "Paired re-analysis" tới hết file
start <- grep("Paired re-analysis", readme)[1]
cat(readme[start:length(readme)], sep = "\n")

## Paired re-analysis with limma

### Motivation
The initial analysis used an unpaired two-sample t-test (`ttest_ind`). However,
GSE10072 contains 33 matched tumor–normal pairs from the same patients
(patient IDs embedded in sample titles), making the samples non-independent.
Treating paired samples as independent mis-estimates the variance. I re-analyzed
the data with a paired design in limma (R) to test whether the original DEG list
was robust to this design–data mismatch.

### Data structure
Of 74 patients: 33 with complete pairs, 25 tumor-only,
16 normal-only (107 samples total). Because 38,3% of samples are
unpaired, the primary model uses all samples with patient as a random effect.

### Methods
- **Primary model:** `lmFit` with `block = patient` and `duplicateCorrelation`
  (consensus within-patient correlation = 0,221), on all 107 samples.
- **Sensitivity model:** fixed-effect paired design `~ patient + tissue` on the
  33 complete pairs (66 samples).
- Probe→gene collapse and 

In [26]:
repo <- "~/python_practice/luad_project"
r <- readLines(file.path(repo, "README.md"))
r <- gsub("—", "-", r)          # thay mọi em-dash bằng hyphen
writeLines(r, file.path(repo, "README.md"))
cat("còn em-dash?", any(grepl("—", r)), "\n")   # kỳ vọng FALSE

Warning message in readLines(file.path(repo, "README.md")):
“incomplete final line found on '~/python_practice/luad_project/README.md'”


còn em-dash? FALSE 


In [28]:
r <- readLines(file.path(repo, "README.md"))        # đọc lại bản mới nhất
cat("còn em-dash?        ", any(grepl("—", r)), "\n")          # FALSE
cat("còn 'inflates'?     ", any(grepl("inflates", r)), "\n")   # FALSE
cat("còn 'ROC'?          ", any(grepl("ROC", r)), "\n")        # FALSE
cat("còn 'Paired design not'?", any(grepl("Paired design not", r)), "\n")  # FALSE
cat("còn dấu phẩy số?    ", any(grepl("0,221|0,9998|38,3", r)), "\n")      # FALSE
cat("có 'addressed'?     ", any(grepl("addressed", r)), "\n")  # TRUE
cat("có 'refined'?       ", any(grepl("refined", r)), "\n")    # TRUE
cat("có 'Paired re-analysis'?", any(grepl("Paired re-analysis", r)), "\n") # TRUE
cat("còn sót ___?        ", any(grepl("___", r)), "\n")        # FALSE

còn em-dash?         FALSE 
còn 'inflates'?      FALSE 
còn 'ROC'?           FALSE 
còn 'Paired design not'? FALSE 
còn dấu phẩy số?     FALSE 
có 'addressed'?      TRUE 
có 'refined'?        TRUE 
có 'Paired re-analysis'? TRUE 
còn sót ___?         FALSE 


In [29]:
start <- grep("Paired re-analysis", r)[1]
cat(r[start:(start+25)], sep = "\n")

## Paired re-analysis with limma

### Motivation
The initial analysis used an unpaired two-sample t-test (`ttest_ind`). However,
GSE10072 contains 33 matched tumor–normal pairs from the same patients
(patient IDs embedded in sample titles), making the samples non-independent.
Treating paired samples as independent mis-estimates the variance. I re-analyzed
the data with a paired design in limma (R) to test whether the original DEG list
was robust to this design–data mismatch.

### Data structure
Of 74 patients: 33 with complete pairs, 25 tumor-only,
16 normal-only (107 samples total). Because 38.3% of samples are
unpaired, the primary model uses all samples with patient as a random effect.

### Methods
- **Primary model:** `lmFit` with `block = patient` and `duplicateCorrelation`
  (consensus within-patient correlation = 0.221), on all 107 samples.
- **Sensitivity model:** fixed-effect paired design `~ patient + tissue` on the
  33 complete pairs (66 samples).
- Probe→gene collapse and 